# Студенческий workbook. Линейная регрессия vs логистическая регрессия

Этот ноутбук повторяет маршрут основного занятия, но в нем больше мест для ваших гипотез, промежуточных выводов и собственных экспериментов.

## Как работать

1. Сначала пишем гипотезу словами.
2. Потом решаем, чем проверять: графиком, таблицей, метрикой.
3. Только после этого запускаем код.
4. В конце обязательно пишем вывод человеческим языком.

## За что можно получить баллы

- `1 балл` за осмысленную гипотезу.
- `1 балл` за связь гипотезы с EDA.
- `1 балл` за корректную проверку.
- `1 балл` за рабочее улучшение модели.
- `+1 балл`, если вы объяснили, почему идея должна была помочь именно этой модели.

In [1]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
# из-за квадратов сильно наказываются выбросы, сложно интерпретировать (просто непонятно как оценить, норм или не норм, из-за квадрата)
def mse_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean((y_true - y_pred) ** 2)

# мягче к выбросам и к ошибкам в целом (не штрафуется квадратом)
def mae_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))

# сохраняет свойство наказания за выбросы, но легче интерпретировать
def rmse_manual(y_true, y_pred):
    return mse_manual(y_true, y_pred) ** 0.5

# отношение разности сумм квадратов в общем и ошибок
def r2_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot else 0.0

# процент отклонения от реальных данных
def mape_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def regression_metrics(y_true, y_pred):
    rmse = rmse_manual(y_true, y_pred)
    std_y = np.std(np.asarray(y_true, dtype=float))
    return {
        "R2": r2_manual(y_true, y_pred),
        "MAE": mae_manual(y_true, y_pred),
        "RMSE": rmse,
        "NRMSE": rmse / std_y if std_y else np.nan,
        "MAPE_%": mape_manual(y_true, y_pred),
    }

# доля правильных ответов
def accuracy_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    return np.mean(y_true == pred)

# соотношение true positive ко всем positive (насколько можем доверять)
def precision_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = np.sum((y_true == 1) & (pred == 1))
    fp = np.sum((y_true == 0) & (pred == 1))
    return tp / (tp + fp) if (tp + fp) else 0.0

# сколько раз мы предсказали tp из всех РЕАЛЬНЫХ true
def recall_manual(y_true, pred):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    tp = np.sum((y_true == 1) & (pred == 1))
    fn = np.sum((y_true == 1) & (pred == 0))
    return tp / (tp + fn) if (tp + fn) else 0.0

# среднее гармоническое между precision и recall. в отличие от обычного среднего, оно сильно наказывает модель, если одна из метрик близка к нулю. используется, когда важно и не пропускать объекты, и не ошибаться в прогнозах
def f1_manual(y_true, pred):
    precision = precision_manual(y_true, pred)
    recall = recall_manual(y_true, pred)
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

# наказывает модель, если она уверенно ошиблась
def logloss_manual(y_true, prob):
    y_true = np.asarray(y_true, dtype=float)
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    return -np.mean(y_true * np.log(prob) + (1 - y_true) * np.log(1 - prob))

def classification_metrics(y_true, prob, pred):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    pred = np.asarray(pred, dtype=int)
    return {
        "Accuracy": accuracy_manual(y_true, pred),
        "Precision": precision_manual(y_true, pred),
        "Recall": recall_manual(y_true, pred),
        "F1": f1_manual(y_true, pred),
        "ROC_AUC": roc_auc_score(y_true, prob),
        "LogLoss": logloss_manual(y_true, prob),
    }

# заполняет трейн и тест медианами
def fill_missing_from_train(X_train, X_test):
    if isinstance(X_train, pd.DataFrame):
        X_train_filled = X_train.copy()
        X_test_filled = X_test.copy()
        numeric_cols = X_train_filled.select_dtypes(include=[np.number]).columns
        medians = X_train_filled[numeric_cols].median()
        X_train_filled[numeric_cols] = X_train_filled[numeric_cols].fillna(medians)
        X_test_filled[numeric_cols] = X_test_filled[numeric_cols].fillna(medians)
        return X_train_filled, X_test_filled

    X_train_filled = np.asarray(X_train).copy()
    X_test_filled = np.asarray(X_test).copy()
    medians = np.nanmedian(X_train_filled, axis=0)
    train_nan = np.where(np.isnan(X_train_filled))
    test_nan = np.where(np.isnan(X_test_filled))
    if len(train_nan[0]) > 0:
        X_train_filled[train_nan] = np.take(medians, train_nan[1])
    if len(test_nan[0]) > 0:
        X_test_filled[test_nan] = np.take(medians, test_nan[1])
    return X_train_filled, X_test_filled

# преобразует массивы данных в матрицы
def to_numpy_2d(X):
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy(dtype=float)
    elif isinstance(X, pd.Series):
        X = X.to_frame().to_numpy(dtype=float)
    else:
        X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return X

# создаёт столбец единиц, чтобы модель дополнительно вычислила w0
def add_bias(X):
    X_arr = to_numpy_2d(X)
    return np.column_stack([np.ones(len(X_arr)), X_arr])

# формула для линейной регрессии
def manual_linear_weights(X_train, y_train):
    Xb = add_bias(X_train)
    y_arr = np.asarray(y_train, dtype=float)
    return np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y_arr

# предсказываем по весам
def manual_linear_predict(X, weights):
    return add_bias(X) @ weights

# стандартизация данных. теперь они одного масштаба (среднее - 0, отклонение - 1)
def standardize_train_test(X_train, X_test):
    X_train_arr = to_numpy_2d(X_train)
    X_test_arr = to_numpy_2d(X_test)
    mean = X_train_arr.mean(axis=0)
    std = X_train_arr.std(axis=0)
    std[std == 0] = 1.0
    return (X_train_arr - mean) / std, (X_test_arr - mean) / std, mean, std

def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))

# сама функция обучения модели, котора steps раз пытается уменьшить ошибку модели
def fit_logistic_manual(X_train, y_train, lr=0.2, steps=1200):
    Xb = add_bias(X_train)
    y_arr = np.asarray(y_train, dtype=float)
    weights = np.zeros(Xb.shape[1], dtype=float)
    history = []
    for step in range(steps):
        prob = sigmoid(Xb @ weights)
        grad = Xb.T @ (prob - y_arr) / len(y_arr)
        weights -= lr * grad
        if step % 100 == 0 or step == steps - 1:
            history.append(logloss_manual(y_arr, prob))
    return weights, history

# предсказание
def predict_logistic_manual(X, weights):
    return sigmoid(add_bias(X) @ weights)


## Блок 1. Boston Housing: быстрый EDA и baseline

Это мягкий вход в тему. Здесь задача понятная: предсказать `price`.

### До запуска кода ответьте словами

- Какой столбец здесь будет целью?
- Какие признаки кажутся самыми понятными человеку?
- Какие признаки уже по названию кажутся рискованными или странными?
- Что стоит проверить первым: пропуски, распределение цели или корреляции?

In [3]:
boston = pd.read_csv("data/BostonHousing.csv").rename(columns={
    "crim": "crime_rate",
    "zn": "large_lots",
    "indus": "industry",
    "chas": "river",
    "nox": "nox_conc",
    "rm": "rooms_avg",
    "age": "old_buildings",
    "dis": "center_dist",
    "rad": "highway",
    "tax": "property",
    "ptratio": "pupil_teacher",
    "b": "black_population",
    "lstat": "lower_status",
    "medv": "price",
})

boston.head()

,crime_rate,large_lots,industry,river,nox_conc,rooms_avg,old_buildings,center_dist,highway,property,pupil_teacher,black_population,lower_status,price
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [4]:
boston_hypotheses = {
    "target": "price",
    "possible_important_features": ["", "", ""],
    "possible_problem_features": ["", "", ""],
    "first_eda_steps": ["", "", ""],
}

boston_hypotheses

{'target': 'price',
 'possible_important_features': ['', '', ''],
 'possible_problem_features': ['', '', ''],
 'first_eda_steps': ['', '', '']}

In [5]:
# TODO 1. Минимальный EDA на Boston.
# Подсказка 1: shape, columns, dtypes, missing, describe.
# Подсказка 2: в Boston есть несколько пропусков в rooms_avg, их стоит заметить.
# Подсказка 3: полезно построить хотя бы:
#   - распределение price
#   - boxplot для crime_rate
#   - корреляции с price
#   - scatter rooms_avg vs price

# Ваш код здесь

In [6]:
print(boston.shape, '\n')

(506, 14) 



In [7]:
print(boston.columns, '\n')

Index(['crime_rate', 'large_lots', 'industry', 'river', 'nox_conc',
       'rooms_avg', 'old_buildings', 'center_dist', 'highway', 'property',
       'pupil_teacher', 'black_population', 'lower_status', 'price'],
      dtype='object') 



In [8]:
print(boston.dtypes, '\n')

crime_rate          float64
large_lots          float64
industry            float64
river                 int64
nox_conc            float64
rooms_avg           float64
old_buildings       float64
center_dist         float64
highway               int64
property              int64
pupil_teacher       float64
black_population    float64
lower_status        float64
price               float64
dtype: object 



In [9]:
print(boston.isna().sum()['rooms_avg'], '\n')

5 



In [10]:
print(boston.describe(), '\n')

       crime_rate  large_lots    industry       river    nox_conc   rooms_avg  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  501.000000   
mean     3.613524   11.363636   11.136779    0.069170    0.554695    6.284341   
std      8.601545   23.322453    6.860353    0.253994    0.115878    0.705587   
min      0.006320    0.000000    0.460000    0.000000    0.385000    3.561000   
25%      0.082045    0.000000    5.190000    0.000000    0.449000    5.884000   
50%      0.256510    0.000000    9.690000    0.000000    0.538000    6.208000   
75%      3.677083   12.500000   18.100000    0.000000    0.624000    6.625000   
max     88.976200  100.000000   27.740000    1.000000    0.871000    8.780000   

       old_buildings  center_dist     highway    property  pupil_teacher  \
count     506.000000   506.000000  506.000000  506.000000     506.000000   
mean       68.574901     3.795043    9.549407  408.237154      18.455534   
std        28.148861     2.105710    8.707

### Вопросы после EDA

- Почему `crime_rate` может быть кандидатом на логарифмирование?
- Почему `rooms_avg` и `lower_status` хочется проверить одними из первых?
- Какие признаки кажутся потенциально коррелированными между собой?

In [11]:
# TODO 2. Постройте baseline для Boston.
# Шаги:
# 1. X = boston.drop(columns='price')
# 2. y = boston['price']
# 3. train_test_split(...)
# 4. обязательно заполните пропуски медианой по train
# 5. обучите LinearRegression
# 6. посчитайте минимум R2, MAE, RMSE
# 7. сравните с predict mean baseline, чтобы метрики не висели в воздухе

# Подсказка:
# y_mean_pred = np.repeat(y_train.mean(), len(y_test))
# regression_metrics(y_test, y_mean_pred)

# Ваш код здесь

In [12]:
boston_conclusion = {
    "is_linear_regression_reasonable_here": "",
    "what_helped_from_eda": "",
    "what_would_you_try_next": ["", ""],
}

boston_conclusion

{'is_linear_regression_reasonable_here': '',
 'what_helped_from_eda': '',
 'what_would_you_try_next': ['', '']}

## Блок 2. Airbnb: делаем учебно-грязную версию

Теперь переходим к более хаотичному датасету.

### Что важно увидеть заранее

- здесь есть текстовые поля;
- есть география;
- есть категориальные признаки;
- есть пропуски;
- есть экстремальные значения.

То есть тут EDA и чистка будут намного важнее.

In [13]:
airbnb_raw = pd.read_csv("data/AB_NYC_2019.csv")

def make_airbnb_dirty(df):
    data = df.copy()
    base_price = data["price"].astype(int).astype(str)
    idx = np.arange(len(data))

    data["price"] = base_price
    data.loc[idx % 3 == 0, "price"] = "$" + base_price[idx % 3 == 0]
    data.loc[idx % 5 == 0, "price"] = base_price[idx % 5 == 0].map(lambda x: f"{int(x):,} USD")
    data.loc[idx % 7 == 0, "price"] = "  " + base_price[idx % 7 == 0] + "  "
    data.loc[idx % 11 == 0, "price"] = base_price[idx % 11 == 0] + ".00 usd"

    data["name"] = data["name"].fillna("")
    data.loc[idx % 6 == 0, "name"] = data.loc[idx % 6 == 0, "name"].str.upper() + "!!!"
    data.loc[idx % 10 == 0, "name"] = "🔥 " + data.loc[idx % 10 == 0, "name"]
    return data

airbnb_dirty = make_airbnb_dirty(airbnb_raw)
airbnb_dirty[["name", "room_type", "price", "minimum_nights", "reviews_per_month"]].sample(8, random_state=42)

,name,room_type,price,minimum_nights,reviews_per_month
879,Come and go as you please in BKLN!,Entire home/apt,$89,3,0.71
44383,"Spacious, sunny room in Queens/Brooklyn",Private room,30,21,NaN
15394,Private bedroom in high-ceiling 4BR apartment!,Private room,120,2,0.43
43230,🔥 SONDER | STOCK EXCHANGE | STUNNING 3BR + KIT...,Entire home/apt,470.00 usd,2,1.88
16332,SPACIOUS 2 BEDROOM WITH BALCONY!!!,Entire home/apt,$199,2,0.80
5966,Entire 2BR APT (not a railroad),Entire home/apt,170,1,3.05
29838,1BR IN SPACIOUS 2 BR IN THE HEART OF WILLIAMSB...,Private room,$90,5,0.06
41764,comfortable Place to live,Entire home/apt,60,30,NaN


In [14]:
airbnb_eda_hypotheses = {
    "type_problems": ["", "", ""],
    "missing_values": ["", ""],
    "outliers": ["", ""],
    "categorical_features": ["", ""],
    "possible_targets": ["price", "room_type"],
}

airbnb_eda_hypotheses

{'type_problems': ['', '', ''],
 'missing_values': ['', ''],
 'outliers': ['', ''],
 'categorical_features': ['', ''],
 'possible_targets': ['price', 'room_type']}

In [15]:
# TODO 3. Проведите EDA на airbnb_dirty.
# Минимум, который стоит сделать:
# 1. посмотреть shape и типы
# 2. посмотреть пропуски
# 3. показать примеры странных значений price
# 4. посмотреть максимум и p99 для minimum_nights
# 5. посмотреть распределение room_type
# 6. построить 2-3 графика

# Подсказка:
# airbnb_dirty.dtypes
# airbnb_dirty.isna().sum().sort_values(ascending=False)
# airbnb_dirty['price'].head(20)
# airbnb_dirty['minimum_nights'].describe()

# Ваш код здесь

### После EDA сформулируйте план чистки

Не пишите код сразу. Сначала словами:

- что сделаем с `price`;
- что сделаем с `name`;
- что сделаем с пропусками;
- что сделаем с экстремальными `minimum_nights`;
- какие новые признаки можно извлечь.

In [16]:
cleaning_plan = {
    "price": "",
    "name": "",
    "missing_values": "",
    "outliers": "",
    "new_features": ["", "", ""],
}

cleaning_plan

{'price': '',
 'name': '',
 'missing_values': '',
 'outliers': '',
 'new_features': ['', '', '']}

In [17]:
# TODO 4. Очистите данные.
# Идеи, которые можно использовать:
# 1. price -> превратить в float
# 2. name -> заполнить пропуски пустой строкой
# 3. reviews_per_month -> заполнить нулями
# 4. last_review_missing -> бинарный признак пропуска
# 5. minimum_nights_clipped -> ограничить сверху, например p99 или 45
# 6. minimum_nights_log -> взять log1p
# 7. name_len, caps_ratio, has_exclamation -> простые текстовые признаки
# 8. is_entire_home -> бинарная цель для логистической регрессии

# Заготовки функций, если хотите:
# def clean_price_series(series):
#     cleaned = series.astype(str).str.replace(r'[^0-9.]', '', regex=True)
#     cleaned = cleaned.replace('', np.nan)
#     return pd.to_numeric(cleaned, errors='coerce')
#
# def caps_ratio(text):
#     letters = [ch for ch in str(text) if ch.isalpha()]
#     if not letters:
#         return 0.0
#     return sum(ch.isupper() for ch in letters) / len(letters)

# Ваш код здесь

## Блок 3. Линейная регрессия на Airbnb: цена

Теперь идем по той же логике, что и в основном ноуте.

### Что важно сделать честно

1. Сначала сравнить с `Mean baseline`.
2. Потом попробовать простую линейную регрессию.
3. Потом добавить feature engineering.
4. Если хочется углубиться, можно посчитать OLS руками через формулу из лекции.

In [18]:
regression_hypotheses = {
    "target": "price или price_capped",
    "baseline_features": ["", "", ""],
    "improved_features": ["", "", "", ""],
    "which_metric_is_most_important": "",
}

regression_hypotheses

{'target': 'price или price_capped',
 'baseline_features': ['', '', ''],
 'improved_features': ['', '', '', ''],
 'which_metric_is_most_important': ''}

In [19]:
# TODO 5. Сделайте baseline для регрессии.
# План:
# 1. выбрать target, например price_capped
# 2. сделать train/test split
# 3. сравнить Mean baseline и sklearn LinearRegression
# 4. посчитать R2, MAE, RMSE, NRMSE, MAPE_%

# Подсказка:
# y_mean_pred = np.repeat(y_train.mean(), len(y_test))
# regression_metrics(y_test, y_mean_pred)
# regression_metrics(y_test, model.predict(X_test))

# Ваш код здесь

In [20]:
# TODO 6. Добавьте feature engineering и сравните качество.
# Идеи признаков:
# - neighbourhood_group
# - room_type
# - latitude, longitude
# - minimum_nights_clipped, minimum_nights_log
# - name_len, caps_ratio, has_exclamation
# - last_review_missing

# Подсказка:
# X_improved = pd.get_dummies(..., drop_first=True)

# Ваш код здесь

In [21]:
# TODO 7. Если хотите пойти глубже, реализуйте ручной OLS.
# Это прямое продолжение формулы из лекции:
# w* = (X^T X)^(-1) X^T y
#
# Подсказка:
# weights = manual_linear_weights(X_train, y_train)
# y_pred_manual = manual_linear_predict(X_test, weights)
# regression_metrics(y_test, y_pred_manual)
#
# Важно:
# - перед этим X должен быть только числовой
# - лучше сравнить ручной результат со sklearn на одной и той же матрице признаков

# Ваш код здесь

### Вопросы после регрессии

- Почему `R2` может быть скромным, а модель всё равно полезнее mean baseline?
- Чем `MAE` отличается от `RMSE` по смыслу?
- Почему `RMSE` особенно чувствительна к редким большим ошибкам?
- Почему на Airbnb вообще не стоит ждать идеально высокой линейной метрики?

## Блок 4. Классификация на Airbnb: тип жилья

Теперь меняем задачу.

### Бинарная цель

- `1` = `Entire home/apt`
- `0` = `Private room`

### Что важно сделать

1. Сравнить с majority baseline.
2. Проверить `LinearRegression + clip` как плохой классификатор.
3. Обучить `LogisticRegression`.
4. Если хочется углубиться, сделать ручную логистическую регрессию через сигмоиду.

In [22]:
classification_hypotheses = {
    "target": "is_entire_home",
    "useful_features": ["", "", "", ""],
    "why_logistic_is_more_natural": "",
    "which_metric_is_most_important": "",
}

classification_hypotheses

{'target': 'is_entire_home',
 'useful_features': ['', '', '', ''],
 'why_logistic_is_more_natural': '',
 'which_metric_is_most_important': ''}

In [23]:
# TODO 8. Сделайте baseline для классификации.
# Шаги:
# 1. оставьте только Entire home/apt и Private room
# 2. соберите признаки
# 3. сделайте majority baseline
# 4. посчитайте Accuracy, Precision, Recall, F1, ROC_AUC, LogLoss

# Подсказка:
# majority_class = int(y_train.mean() >= 0.5)
# majority_prob = np.repeat(float(y_train.mean()), len(y_test))
# majority_pred = np.repeat(majority_class, len(y_test))
# classification_metrics(y_test, majority_prob, majority_pred)

# Ваш код здесь

In [24]:
# TODO 9. Сравните LinearRegression + clip и LogisticRegression.
# Подсказка:
# 1. для LinearRegression получите сырые scores
# 2. потом prob = np.clip(scores, 0, 1)
# 3. pred = (prob >= 0.5).astype(int)
# 4. для LogisticRegression используйте predict_proba
# 5. сравните метрики не только между собой, но и с majority baseline

# Ваш код здесь

In [25]:
# TODO 10. Если хотите пойти глубже, реализуйте ручную логистическую регрессию.
# Это прямое продолжение лекции:
# p(x) = sigmoid(w^T x)
# log-loss = -mean(y log p + (1-y) log(1-p))
#
# Подсказка:
# 1. возьмите маленький набор числовых признаков
# 2. стандартизируйте train/test
# 3. weights, history = fit_logistic_manual(X_train_std, y_train, lr=0.2, steps=1200)
# 4. prob = predict_logistic_manual(X_test_std, weights)
# 5. pred = (prob >= 0.5).astype(int)
# 6. classification_metrics(y_test, prob, pred)

# Ваш код здесь

### Вопросы после классификации

- Почему `Accuracy` не всегда достаточно?
- Когда важнее `Precision`, а когда `Recall`?
- Почему `LogLoss` может быть содержательнее, чем просто доля угаданных классов?
- Если `LinearRegression + clip` не сильно хуже по `Accuracy`, почему мы всё равно говорим, что логистическая регрессия здесь естественнее?

## Блок 5. Свои идеи

Теперь можно предлагать собственные улучшения.

Хорошие направления:

- взять `log(price)` вместо `price_capped`;
- попробовать новые текстовые признаки;
- сильнее ограничить выбросы;
- оставить только крупные районы;
- попробовать другие наборы признаков для ручной логистической регрессии;
- сравнивать новые идеи не только с лучшей моделью, но и с baseline.

In [26]:
student_experiment = {
    "idea": "",
    "why_it_should_help": "",
    "which_metric_should_improve": "",
    "result": "",
}

student_experiment

{'idea': '',
 'why_it_should_help': '',
 'which_metric_should_improve': '',
 'result': ''}

In [27]:
# TODO 11. Песочница для вашей идеи.
# Совет:
# 1. сначала напишите гипотезу словами выше
# 2. потом укажите, по какой метрике ждете улучшение
# 3. только потом пишите код

# Ваш код здесь

## Финальная рефлексия

Заполните коротко, но честно. Это важнее, чем просто много кода.

In [28]:
final_reflection = {
    "what_helped_in_eda_the_most": "",
    "which_regression_metric_was_most_informative": "",
    "which_classification_metric_was_most_informative": "",
    "where_linear_regression_was_natural": "",
    "why_logistic_regression_was_more_natural_for_binary_target": "",
    "which_formula_from_the_lecture_became_clearer_after_code": "",
}

final_reflection

{'what_helped_in_eda_the_most': '',
 'which_regression_metric_was_most_informative': '',
 'which_classification_metric_was_most_informative': '',
 'where_linear_regression_was_natural': '',
 'why_logistic_regression_was_more_natural_for_binary_target': '',
 'which_formula_from_the_lecture_became_clearer_after_code': ''}